# SAFRAME COCO proxy-3 ST-YOLOX 320
Run with a Colab GPU runtime. Execute Setup and Prepare once per runtime, then Smoke, Full training, or Resume. Only `DRIVE_ROOT` is user-configurable. COCO remains on ephemeral storage; models and reports are written to Drive.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/saframe-proxy3'  # Edit only this value
MODELZOO_COMMIT = '0f6210ed5156126b782e1c43249063a477484b20'
CLASS_NAMES = ('person', 'book', 'stop sign')


In [ ]:
# Setup: Drive, GPU/disk preflight, pinned Model Zoo Services and dependencies.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys
free_gib = shutil.disk_usage('/content').free / 2**30
if free_gib < 45:
    raise RuntimeError(f'At least 45 GiB free space is required; found {free_gib:.1f} GiB')
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
mz = Path('/content/stm32ai-modelzoo-services')
if not mz.exists():
    subprocess.run(['git', 'clone', 'https://github.com/STMicroelectronics/stm32ai-modelzoo-services.git', str(mz)], check=True)
subprocess.run(['git', '-C', str(mz), 'fetch', '--depth=1', 'origin', MODELZOO_COMMIT], check=True)
subprocess.run(['git', '-C', str(mz), 'checkout', '--detach', MODELZOO_COMMIT], check=True)
# Model Zoo 4.1.1 pins TensorFlow 2.18, whose wheels support Python 3.9-3.12, not 3.13.
uv = shutil.which('uv')
if uv is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    uv = shutil.which('uv')
if uv is None:
    raise RuntimeError('uv installation succeeded but its executable is not on PATH')
venv = Path('/content/saframe-modelzoo-py311')
MZ_PYTHON = str(venv / 'bin' / 'python')
if not Path(MZ_PYTHON).exists():
    subprocess.run([uv, 'python', 'install', '3.11'], check=True)
    subprocess.run([uv, 'venv', '--python', '3.11', str(venv)], check=True)
subprocess.run([uv, 'pip', 'install', '--python', MZ_PYTHON, '-r', str(mz / 'requirements.txt')], check=True)
headless_env = dict(os.environ, MPLBACKEND='Agg')
probe = subprocess.run([MZ_PYTHON, '-c', 'import tensorflow as tf; print(tf.__version__); print(tf.config.list_physical_devices("GPU")); assert tf.config.list_physical_devices("GPU")'], text=True, capture_output=True, env=headless_env)
print(probe.stdout, end='')
if probe.returncode != 0:
    print(probe.stderr, file=sys.stderr)
    raise RuntimeError('TensorFlow import or GPU probe failed; inspect stderr above.')
print(f'Free disk: {free_gib:.1f} GiB')


In [ ]:
# Prepare COCO in ephemeral storage and create clipped, contiguous proxy-3 labels.
import json, os, urllib.request, zipfile
from collections import Counter
work = Path('/content/saframe-proxy3')
downloads = work / 'downloads'
coco = work / 'coco'
proxy = work / 'proxy3'
downloads.mkdir(parents=True, exist_ok=True)
urls = {
 'train2017.zip': 'https://images.cocodataset.org/zips/train2017.zip',
 'val2017.zip': 'https://images.cocodataset.org/zips/val2017.zip',
 'annotations_trainval2017.zip': 'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
}
for filename, url in urls.items():
    archive = downloads / filename
    if not archive.exists():
        subprocess.run(['wget', '-c', url, '-O', str(archive)], check=True)
    marker = coco / (filename + '.extracted')
    if not marker.exists():
        with zipfile.ZipFile(archive) as stream: stream.extractall(coco)
        marker.touch()
targets = ((1, 1, 'person'), (84, 2, 'book'), (13, 3, 'stop sign'))
def convert(split):
    source = json.loads((coco / 'annotations' / f'instances_{split}2017.json').read_text())
    images = {item['id']: item for item in source['images']}
    mapping = {source_id: target_id for source_id, target_id, _ in targets}
    selected, used, counts = [], set(), Counter()
    for ann in source['annotations']:
        if ann['category_id'] not in mapping: continue
        image = images[ann['image_id']]
        x, y, width, height = map(float, ann['bbox'])
        x0, y0 = max(0., min(image['width'], x)), max(0., min(image['height'], y))
        x1, y1 = max(0., min(image['width'], x + width)), max(0., min(image['height'], y + height))
        if x1 <= x0 or y1 <= y0: continue
        item = dict(ann, category_id=mapping[ann['category_id']], bbox=[x0, y0, x1-x0, y1-y0], area=(x1-x0)*(y1-y0))
        selected.append(item); used.add(item['image_id']); counts[item['category_id']] += 1
    out = proxy / split; out.mkdir(parents=True, exist_ok=True)
    grouped = {image_id: [] for image_id in used}
    for ann in selected: grouped[ann['image_id']].append(ann)
    for image_id in sorted(used):
        image = images[image_id]; src = (coco / f'{split}2017' / image['file_name']).resolve(); dst = out / image['file_name']
        if not dst.exists(): dst.symlink_to(os.path.relpath(src, dst.parent))
        rows = []
        for ann in grouped[image_id]:
            x, y, width, height = ann['bbox']; cls = ann['category_id'] - 1
            rows.append(f'{cls} {(x+width/2)/image["width"]:.8f} {(y+height/2)/image["height"]:.8f} {width/image["width"]:.8f} {height/image["height"]:.8f}')
        dst.with_suffix('.txt').write_text('\n'.join(rows) + '\n')
    summary = {'class_names': list(CLASS_NAMES), 'images': len(used), 'annotations': len(selected), 'annotations_by_class': {CLASS_NAMES[i]: counts[i+1] for i in range(3)}}
    (Path(DRIVE_ROOT) / f'{split}_dataset_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
    print(split, summary)
    return out
train_dir, val_dir = convert('train'), convert('val')
for root in (train_dir, val_dir):
    seen = Counter()
    for label in root.glob('*.txt'):
        rows = label.read_text().splitlines()
        if not rows: raise RuntimeError(f'Empty label: {label}')
        for row in rows:
            fields = row.split(); cls = int(fields[0]); coords = list(map(float, fields[1:]))
            if len(fields) != 5 or cls not in range(3) or not all(0 <= x <= 1 for x in coords): raise RuntimeError(f'Invalid label: {label}: {row}')
            seen[cls] += 1
    if set(seen) != {0, 1, 2}: raise RuntimeError(f'Missing class in {root}: {seen}')
print('Dataset contract: PASS')


In [ ]:
# Write the pinned training contract and execution helper.
config_dir = Path('/content/saframe-config'); config_dir.mkdir(exist_ok=True)
config = '''general:
  project_name: saframe_coco_proxy3_st_yoloxn_320
  logs_dir: logs
  saved_models_dir: saved_models
  display_figures: false
  gpu_memory_limit: 16
  num_threads_tflite: 12
  global_seed: 127
model:
  model_type: st_yoloxn
  model_name: st_yoloxn_d033_w025
  input_shape: (320,320,3)
  pretrained: false
operation_mode: chain_tqe
dataset:
  dataset_name: darknet_yolo
  format: darknet_yolo
  download_data: false
  exclude_unlabeled: true
  max_detections: 100
  class_names: [person, book, stop sign]
  train_images_path: ${oc.env:PROXY3_TRAIN_DIR}
  train_annotations_path: ${oc.env:PROXY3_TRAIN_DIR}
  val_images_path: ${oc.env:PROXY3_VAL_DIR}
  val_annotations_path: ${oc.env:PROXY3_VAL_DIR}
  test_images_path: ${oc.env:PROXY3_VAL_DIR}
  test_annotations_path: ${oc.env:PROXY3_VAL_DIR}
  quantization_path: ${oc.env:PROXY3_TRAIN_DIR}
  quantization_split: 0.001
preprocessing:
  rescaling: {scale: 1/255, offset: 0}
  resizing: {aspect_ratio: fit, interpolation: nearest}
  color_mode: rgb
data_augmentation:
  random_crop: {crop_center_x: (0.25, 0.75), crop_center_y: (0.25, 0.75), crop_width: (0.6, 0.9), crop_height: (0.6, 0.9), change_rate: 0.5}
  random_periodic_resizing: {period: 20, image_sizes: [(224,224), (256,256), (288,288), (320,320), (352,352), (384,384), (416,416)]}
  random_contrast: {factor: 0.4}
  random_brightness: {factor: 0.3}
  random_flip: {mode: horizontal}
  random_rotation: {factor: 0.02, fill_mode: wrap, interpolation: nearest}
training:
  resume_training: false
  dropout: null
  batch_size: 64
  epochs: 500
  optimizer:
    Adam: {learning_rate: 0.0025}
  callbacks:
    LRWarmupCosineDecay: {initial_lr: 1.0e-05, warmup_steps: 20, max_lr: 0.00125, hold_steps: 20, decay_steps: 350, end_lr: 1.0e-06}
    EarlyStopping: {monitor: val_loss, patience: 60, restore_best_weights: true, verbose: 1}
postprocessing: {confidence_thresh: 0.001, NMS_thresh: 0.5, IoU_eval_thresh: 0.5, plot_metrics: true, max_detection_boxes: 100}
quantization: {quantizer: TFlite_converter, quantization_type: PTQ, quantization_input_type: uint8, quantization_output_type: int8, export_dir: quantized_models}
mlflow: {uri: ./tf/src/experiments_outputs/mlruns}
hydra:
  run:
    dir: ./tf/src/experiments_outputs/${now:%Y_%m_%d_%H_%M_%S}
'''
(config_dir / 'proxy3_colab.yaml').write_text(config)
def run_training(output_name, epochs=500, resume=False, train_path=train_dir, val_path=val_dir):
    output = Path(DRIVE_ROOT) / 'experiments' / output_name; output.mkdir(parents=True, exist_ok=True)
    (output / 'effective_config.yaml').write_text(config)
    env = dict(os.environ, MPLBACKEND='Agg', PROXY3_TRAIN_DIR=str(train_path), PROXY3_VAL_DIR=str(val_path))
    command = [MZ_PYTHON, 'stm32ai_main.py', '--config-path', str(config_dir), '--config-name', 'proxy3_colab.yaml', f'training.epochs={epochs}', f'hydra.run.dir={output}']
    if resume:
        last_model = output / 'saved_models' / 'last_model.keras'
        if not last_model.is_file(): raise FileNotFoundError(last_model)
        command += ['training.resume_training=true', f'model.model_path={last_model}']
    subprocess.run(command, cwd=mz / 'object_detection', env=env, check=True)


In [ ]:
# Smoke: one epoch plus PTQ/evaluation on a deterministic subset.
def subset(source, destination, limit):
    destination.mkdir(parents=True, exist_ok=True); selected = []; seen = set()
    for label in sorted(source.glob('*.txt')):
        classes = {int(row.split()[0]) for row in label.read_text().splitlines()}
        if len(selected) < limit or seen != {0,1,2}: selected.append(label); seen.update(classes)
        if len(selected) >= limit and seen == {0,1,2}: break
    for label in selected:
        image = next(path for path in source.glob(label.stem + '.*') if path.suffix.lower() in {'.jpg','.jpeg','.png'})
        for item in (label, image):
            target = destination / item.name
            if not target.exists(): target.symlink_to(item.resolve())
    return destination
smoke_train = subset(train_dir, work / 'smoke_train', 256)
smoke_val = subset(val_dir, work / 'smoke_val', 64)
run_training('smoke', epochs=1, train_path=smoke_train, val_path=smoke_val)


In [ ]:
# Full baseline: run only after Smoke succeeds.
run_training('full', epochs=500)


In [ ]:
# Resume the interrupted full run after repeating Setup and Prepare.
run_training('full', epochs=500, resume=True)


In [ ]:
# Inventory persistent deliverables without copying COCO to Drive.
import hashlib
root = Path(DRIVE_ROOT) / 'experiments' / 'full'
artifacts = []
for path in sorted(root.rglob('*')):
    if path.is_file() and path.suffix.lower() in {'.keras', '.h5', '.tflite', '.json', '.csv', '.png', '.yaml', '.npz'}:
        artifacts.append({'path': str(path.relative_to(root)), 'bytes': path.stat().st_size, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
manifest = {'modelzoo_commit': MODELZOO_COMMIT, 'class_names': list(CLASS_NAMES), 'artifacts': artifacts}
(root / 'artifact_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(json.dumps(manifest, indent=2))
